In [1]:
!cp -r /kaggle/input/datasets/baohg153/prover9/Prover9-LADR-2026-4F /kaggle/working/

%cd /kaggle/working/Prover9-LADR-2026-4F
!make all STATIC=1
%cd /kaggle/working/

/kaggle/working/Prover9-LADR-2026-4F
cd ladr         && make lib
make[1]: Entering directory '/kaggle/working/Prover9-LADR-2026-4F/ladr'
make libladr.a
make[2]: Entering directory '/kaggle/working/Prover9-LADR-2026-4F/ladr'
ladr: build mode changed to release, rebuilding...
gcc -O2 -Wall -MMD -MP     -c -o order.o order.c
gcc -O2 -Wall -MMD -MP     -c -o clock.o clock.c
gcc -O2 -Wall -MMD -MP     -c -o nonport.o nonport.c
gcc -O2 -Wall -MMD -MP     -c -o fatal.o fatal.c
gcc -O2 -Wall -MMD -MP     -c -o ibuffer.o ibuffer.c
gcc -O2 -Wall -MMD -MP     -c -o memory.o memory.c
gcc -O2 -Wall -MMD -MP     -c -o hash.o hash.c
gcc -O2 -Wall -MMD -MP     -c -o string.o string.c
gcc -O2 -Wall -MMD -MP     -c -o strbuf.o strbuf.c
gcc -O2 -Wall -MMD -MP     -c -o glist.o glist.c
gcc -O2 -Wall -MMD -MP     -c -o options.o options.c
gcc -O2 -Wall -MMD -MP     -c -o symbols.o symbols.c
gcc -O2 -Wall -MMD -MP     -c -o avltree.o avltree.c
gcc -O2 -Wall -MMD -MP     -c -o term.o term.c
gcc -O2 -Wall -MM

In [2]:
!pip install pandas \
            seaborn \
            matplotlib \
            nltk \
            regex \
            torch \
            datasets \
            transformers \
            peft \
            nltk

In [3]:
import torch
import json
from datasets import Dataset,DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, PeftModel
from torch.nn import CrossEntropyLoss

import nltk
from nltk.sem.logic import LogicParser
from nltk.inference import Prover9
import regex as re  
from typing import Optional
import os
import gc
from tqdm import tqdm
from torch.utils.data import DataLoader

In [4]:
class ManualDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "id": item["id"],
            "premises": item["premises"],
            "conclusion": item["conclusion"],
            "label": item["label"]
        }

In [5]:
class FOLModel:
    def __init__(
        self,
        model_name: str = "/kaggle/input/models/ductri0981/deepseek-model/transformers/default/1/deepseek_model",
        output_dir: str = "/kaggle/working/fol_model",
        max_length: int = 768,
        local_files_only=False
    ):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        
        #Load tokenizer
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            local_files_only=local_files_only
        )
        #Load base model.
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True,
            local_files_only=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

        self.model.config.use_cache = False
        
    def load_finetune_model(self, path_adapter: str):
        self.model = PeftModel.from_pretrained(
            self.model,
            path_adapter,
            local_files_only=True
        )
        print("Load adapter successfully")

        self.model = self.model.merge_and_unload()
    
        print("Merged LoRA into base model")
    
        for param in self.model.parameters():
            param.requires_grad = False
    
        self.model.config.use_cache = False

    def build_prompt(self, premises, conclusion):
        sentence = premises + " " + conclusion
        formatted = [
            s.strip() for s in sentence.split('.') if s.strip()
        ]
        formatted = "\n".join(
            [f"{i+1}. {s}." for i, s in enumerate(formatted)]
        )
        
        prompt = f"""### Role
You are a precise Natural Language to First-Order Logic (FOL) translator. Your task is to perform a direct, line-by-line mapping of sentences into formal logic.

### Logical Symbols
Strictly use these symbols:
- Universal: ∀
- Existential: ∃
- Conjunction: ∧
- Disjunction: ∨
- Exclusive Or: ⊕
- Negation: ¬
- Implication: →
- Equivalence: ↔

### Strict Constraints
1. **One-to-One Parity**: The number of FOL expressions in the output must exactly match the number of sentences in the Natural Language input. 
2. **Line Separation**: Each FOL expression must be placed on a new line. Do not use any other delimiters or numbering.
3. **Predicate Consistency**: 
   - Use a single, consistent name for each property or relation. 
   - Do not use synonyms (e.g., if you use "Student(x)", do not use "is_student(x)" or "IsStudent(x)" elsewhere).
   - Prefer concise noun-based or verb-based naming (e.g., "Teacher(x)", "Loves(x, y)").
4. **No Metadata**: Output ONLY the FOL expressions. No explanations, no thought process, and no introductory remarks.

Now apply the rules strictly to the following input:

### Input:
{formatted}

### Output:
"""
        return prompt
        
    def predict(self, loader, max_new_tokens: int = 768, num_outputs = 3):
        self.model.eval()

        all_outputs = []
        all_ids = []
        data = []
        with torch.no_grad():
            loop = tqdm(loader)
            for batch in loop:

                ids = batch["id"]
                premises = batch["premises"]
                conclusions = batch["conclusion"]
                labels = batch["label"]
                # ensure list
                if isinstance(premises, str):
                    premises = [premises]
                    conclusions = [conclusions]
                    ids = [ids]

                prompts = []
                for p, c in zip(premises, conclusions):
                    prompts.append(self.build_prompt(p, c))

                inputs = self.tokenizer(
                    prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=self.max_length
                ).to(self.model.device)

                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=1,
                    min_p=0.05,
    
                    num_return_sequences=num_outputs,
                    pad_token_id=self.tokenizer.eos_token_id
                )

                decoded_outputs = self.tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True
                )
    
                batch_size = len(prompts)
    
                for i in range(batch_size):
    
                    sample_outputs = []
    
                    for j in range(num_outputs):
    
                        idx = i * num_outputs + j
                        text = decoded_outputs[idx]
    
                        if "### Output:" in text:
                            text = text.split("### Output:")[-1].strip()
                        else:
                            text = text[len(prompts[i]):].strip()
    
                        sample_outputs.append(text)
    
                    data.append({
                        "id": ids[i],
                        "natural": premises[i] + '\n' + conclusions[i],
                        "predictions": [
                            {"fol": fol, "label": None}
                            for fol in sample_outputs
                        ],
                        "label": labels[i]
                    })
        return data

In [6]:
class DataFilter:
    '''
    A class used to filter First-Order Logic (FOL) predictions within a dataset entry.
    It deduplicates identical FOL strings and ensures logical syntax integrity by 
    checking for balanced brackets (crucial for catching truncated LLM outputs).
    
    Attributes
    ----------
    - duplicate_count : int
        Counter for the number of duplicate predictions removed.
    
    - syntax_error_count : int
        Counter for the number of predictions with invalid/unbalanced syntax.
        
    Methods
    -------
    - _is_valid_syntax(fol_str)
        Uses a stack-based approach to verify balanced parentheses, brackets, and braces.
        
    - filter(entry)
        Processes a single data entry, filtering its 'predictions' list.
    '''
    
    def __init__(self):
        self.duplicate_count = 0
        self.syntax_error_count = 0
        self.length_ratio_count=0
    
    def _is_valid_syntax(self, fol_str: str) -> bool:
        '''
        Validates that all opening brackets/parentheses have corresponding closing pairs.
        
        Parameters
        ----------
        - fol_str : str
            The FOL string to validate.
            
        Returns
        -------
        - bool
            True if syntax is balanced, False otherwise.
        '''
        if not fol_str:
            return False
            
        stack = []
        matching_bracket = {')': '(', ']': '[', '}': '{'}
        
        for char in fol_str:                   
            if char in matching_bracket.values():
                stack.append(char)
            elif char in matching_bracket.keys():
                if not stack or stack.pop() != matching_bracket[char]:
                    return False

        return len(stack) == 0
    
    
    def _is_valid_length_ratio(self, nat_str: str, fol_str: str, lowerbound_ratio: float=0.5, upperbound_ratio: float=2.0) -> bool:
        '''
        Validates that the ratio of the FOL string length to the natural string length falls within a specified range.
        
        The ratio is calculated as: ratio = len(fol_str) / len(nat_str)}

        Parameters
        ----------
        nat_str : str
            The natural language string to reference.
        fol_str : str
            The fol string to validate.
        lowerbound_ratio : float, optional
            The inclusive lower bound for the ratio (default is 0.5).
        upperbound_ratio : float, optional
            The inclusive upper bound for the ratio (default is 2.0).
            
        Returns
        -------
        bool
            True if the ratio is within [lowerbound_ratio, upperbound_ratio], False otherwise.
            Returns False if nat_str is empty to avoid division by zero.
        '''
        nat_length = len(nat_str)
        if nat_length == 0:
            return False
    
        fol_length = len(fol_str)
        ratio = fol_length / nat_length
        
        return lowerbound_ratio <= ratio <= upperbound_ratio
    
    
    def filter(self, entry: dict, lowerbound_ratio:float=0.5, upperbound_ratio:float=2.0) -> dict:
        '''
        Filters the 'predictions' list of a single entry by removing duplicates 
        and syntactically broken FOL strings.
        
        Parameters
        ----------
        - entry : dict
            A data point containing a "predictions" key with a list of FOL dicts.
            
        Returns
        -------
        - dict
            A copy of the entry containing only valid, unique predictions.
        '''
        seen_fols = set()
        valid_predictions = []
        nat_str = entry.get("natural", "")
        
        natural_text = entry.get("natural", "").strip()
        sentences = [s for s in natural_text.split('.') if s.strip()]
        nat_length = len(sentences)
        
        for pred in entry.get("predictions", []):
            fol = pred.get("fol", "")
            
            # Check for duplicates within this specific entry
            if fol in seen_fols:
                continue
            
            # Validate syntax (bracket balancing)
            if not self._is_valid_syntax(fol):
                continue
            
            premise_list = [p.strip() for p in fol.split('\n') if p.strip()]
            unique_premises = []
            premise_set = set()
            
            for premise in premise_list:
                if premise not in premise_set:
                    unique_premises.append(premise)
                    premise_set.add(premise)
                
            if len(unique_premises) != nat_length:
                continue
            
            # Check for length ratio
            if not self._is_valid_length_ratio(nat_str, fol, lowerbound_ratio, upperbound_ratio):
                self.length_ratio_count += 1
                continue
            
            seen_fols.add(fol)
            valid_predictions.append(pred)
            
        filtered_element = entry.copy()
        filtered_element["predictions"] = valid_predictions
        
        return filtered_element

def filter2(fol_predictions, ground_truth, threshold: float = 0.5):
    if len(fol_predictions) == 0:
        return None

    count = {"True": 0, "False": 0, "Uncertain": 0}

    for prediction in fol_predictions:
        count[prediction["label"]] += 1

    if count[ground_truth] / len(fol_predictions) > threshold:
        return True
    return False

In [7]:
class Engine:
    def __init__(self, prover9_path='C:\\Program Files (x86)\\Prover9-Mace4\\bin-win32'):
        self.parser = LogicParser()
        self.prover9_path = prover9_path
        self.prover = Prover9()
        self.prover.config_prover9(prover9_path)
    
    def _translate_fol(self, fol_text: str):
        # '-' --> '_'
        fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

        replacements = {
            '∀': 'all ', 
            '∃': 'exists ',
            '∧': '&', 
            '∨': '|',
            '⊕': '^',
            '¬': '-',
            '→': '->', 
            '⟷': '<->',
            '↔': '<->'
        }
        for k, v in replacements.items():
            fol_text = fol_text.replace(k, v)

        # Add dot: "all x (P(x))" --> "all x. (P(x))"
        fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

        # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
        words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
        reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
        
        for w in set(words):
            if w not in reserved_words:
                fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
        return fol_text


    def _is_valid_fol(self, fol_list):
        try:
            for line in fol_list:
                if line.strip():
                    self.parser.parse(line)
            return True
        except Exception:
            return False

    def _check_conclusion(self, premises, conclusion):
        '''
        This function is the engine. It checks whether the conclusion is True/False/Uncertain based on the list of premises.
        Args:
            premises: list[str]
            conclusion: str
        Returns: 
            str ("True"/"False"/"Uncertain")
        '''
        # Read fol strings
        translated_premises = [self._translate_fol(p) for p in premises]
        translated_conclusion = self._translate_fol(conclusion)

        if (not self._is_valid_fol(translated_premises) or not self._is_valid_fol([translated_conclusion])):
            error_msg = f"Invalid FOL syntax detected!"
            raise ValueError(error_msg)
        
        try:
            parsed_premises = [self.parser.parse(p) for p in translated_premises]
            parsed_conclusion = self.parser.parse(translated_conclusion)
        except Exception as e:
            print(e)
            raise f"Error: {e}"

        # Check conclusion
        if self.prover.prove(parsed_conclusion, []):
            raise Exception("Error: The conclusion itself is True!")
        elif self.prover.prove(parsed_conclusion.negate(), []):
            raise Exception("Error: The conclusion itself is False!")
        elif not self.prover.prove(None, parsed_premises):
            is_true = self.prover.prove(parsed_conclusion, parsed_premises)
            if is_true:
                return "True"

            is_false = self.prover.prove(parsed_conclusion.negate(), parsed_premises)
            if is_false:
                return "False"

            return "Uncertain"
        else: 
            raise Exception("Error: The premises are inconsistent!")
    
    def check_conclusion(self, data):
        fol_list = data["predictions"]
        new_fol_list = []
        for fol in fol_list:
            try:
                premises = fol["fol"].split("\n")[:-1]
                conclusion = fol["fol"].split("\n")[-1]
                result = self._check_conclusion(premises, conclusion)

                if result == "True":
                    fol["label"] = "True"
                elif result == "False":   # FIX 2
                    fol["label"] = "False"
                else:
                    fol["label"] = "Uncertain"
                new_fol_list.append(fol)
            except:
                continue

        data["predictions"] = new_fol_list
        return data

In [8]:
import json
with open("/kaggle/input/datasets/baohg153/folio-test/test_data.json", 'r', encoding="utf-8") as f:
    dataset = json.load(f)
dataset = ManualDataset(dataset)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
)

fol_model = FOLModel(
    model_name="/kaggle/input/models/ductri0981/deepseek-model/transformers/default/1/deepseek_model",
    local_files_only=True
)

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [9]:
fol_model.load_finetune_model("/kaggle/input/models/ductri0981/fol-model/transformers/default/2")

dataset_formated_predicted = fol_model.predict(loader)

Load adapter successfully
Merged LoRA into base model


100%|██████████| 3/3 [00:59<00:00, 19.67s/it]


In [10]:
engine = Engine(prover9_path="/kaggle/working/Prover9-LADR-2026-4F/bin")
filter_1 = DataFilter()

# Bao
before_filter_preds = []

dataset_fine_tune = []
count_correct = 0
for fol in dataset_formated_predicted:
    before_filter_preds.append(fol) # Bao
    
    fol = filter_1.filter(fol)
    fol = engine.check_conclusion(fol)
    result_filter2 = filter2(fol['predictions'], fol["label"])
    if result_filter2:
        count_correct += 1
    dataset_fine_tune.append(fol)

accuracy = count_correct / len(dataset_fine_tune)
results = {
    "accuracy": f"{accuracy:.4f}",
    "data": dataset_fine_tune
}
with open("result_basic_2.json", 'w', encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

with open("preds_before_finetune.json", 'w', encoding="utf-8") as f:
    json.dump(before_filter_preds, f, ensure_ascii=False, indent=4)